In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects
from scipy import signal
from scipy.stats import norm

SOURCE_FILE = Path('/home/data/zml/testcarfind/下午/ch1_2026-09-05-13-51-27_2_30720_520_1000.bin')
WINDOW_START, WINDOW_END = 0.0, 60.0  # 相对于所选文件的秒数
PARAMETERS = dict(
    channel_from=291, channel_to=520, channel_spacing=4.0,
    frequency_low=0.01, frequency_high=1.0,
    peak_prominence=2.5, minimum_separation=2.0, prominence_window=6.0,
    seed_sigma=0.1, seed_width=20, seed_stride=50,
    tracking_tolerance=1, maximum_missed_channels=20,
    sigma_a=0.01,
    minimum_track_channels=50, minimum_speed=0.0, maximum_speed=120 / 3.6,
    direction='auto', polarity='both',
)


In [ ]:
def read_window(source_file, window_start, window_end, padding_seconds):  # 读取目标窗口，并在前后各多读一段时间
    files = sorted(source_file.parent.glob('*.bin'))  # 找到同目录所有 bin 文件，按文件名排序
    headers = [np.fromfile(path, dtype='<f4', count=20) for path in files]  # 每个文件读取前 20 个小端 float32，共 80 字节
    anchor_index = files.index(source_file)  # 找到参考文件在列表中的位置
    frames = np.array([int(header[7]) for header in headers])  # 文件头第 8 个数：该文件每通道的采样点数,30720,30.72S
    num_channels = int(headers[anchor_index][8])  # 参考文件头第 9 个数：通道数
    sampling_rate = float(headers[anchor_index][9])  # 参考文件头第 10 个数：采样率，单位 Hz
    boundaries = np.r_[0, np.cumsum(frames)]  # 将文件视为连续数据，计算各文件起止采样编号
    anchor_sample = boundaries[anchor_index]  # 参考文件开头在全部数据中的采样编号
    first = anchor_sample + round((window_start - padding_seconds) * sampling_rate) # 读取起点：目标起点向前扩展，最早到全部数据开头
    last = anchor_sample + round((window_end + padding_seconds) * sampling_rate) # 读取终点：目标终点向后扩展，最晚到全部数据末尾
    blocks = []  # 保存从各文件取出的数据片段
    for index, path in enumerate(files):  # 依次检查每个文件
        left, right = max(first, boundaries[index]), min(last, boundaries[index + 1])  # 求读取范围与当前文件的交集
        if left < right:  # 有交集才需要读取当前文件
            mapped = np.memmap(path, dtype='<f4', mode='r', offset=80,  # 只读映射文件，跳过 80 字节文件头
                               shape=(num_channels, frames[index]))  # 按“通道数 × 时间采样点数”解释数据
            print(mapped.shape)  # 打印映射形状、读取范围、当前文件的起止编号
            blocks.append(mapped[:, left - boundaries[index]:right - boundaries[index]])  # 整体编号转为文件内编号，截取所有通道的数据
    return np.concatenate(blocks, axis=1), sampling_rate, int(first - anchor_sample)  # 返回拼接数据、采样率、起点相对参考文件的采样偏移


raw_data, sampling_rate, sample_origin = read_window(  # 接收读取结果
    SOURCE_FILE, WINDOW_START, WINDOW_END, 1 / PARAMETERS['frequency_low'])  # 前后各补充“最低频率的一个周期”
print(f'读取 {raw_data.shape}，采样率 {sampling_rate:g} Hz')  # 将起点采样偏移换算为秒并打印

In [ ]:
def preprocess(raw_data, sampling_rate, parameters):
    # 1. 去趋势 - 去除长时间漂移和直流偏移
    from daspy.basic_tools.preprocessing import detrending
    detrended = np.empty_like(raw_data, dtype=np.float32)
    for i in range(len(raw_data)):
        detrended[i] = detrending(raw_data[i])
    
    # 2. 带通滤波
    frequencies = np.array([parameters['frequency_low'], parameters['frequency_high']])
    sos = signal.iirfilter(4, frequencies / (sampling_rate / 2),
                          btype='bandpass', ftype='butter', output='sos')
    filtered = np.empty_like(detrended, dtype=np.float32)
    for first in range(0, len(detrended), 8):
        block = signal.sosfilt(sos, detrended[first:first + 8], axis=1)
        filtered[first:first + 8] = signal.sosfilt(sos, block[:, ::-1], axis=1)[:, ::-1]
    
    # 3. 共模噪声去除
    if parameters.get('remove_common', True):
        common = np.median(filtered, axis=0)
        energy = float(np.sum(common ** 2))
        if energy > np.finfo(float).eps:
            projection = np.sum(filtered * common, axis=1) / energy
            filtered -= projection[:, None] * common
    
    # 4. 通道选择
    data = filtered[parameters['channel_from'] - 1:parameters['channel_to']].astype(float)
    
    # 5. MAD归一化
    centered = data - np.median(data, axis=1, keepdims=True)
    mad = np.median(np.abs(centered), axis=1, keepdims=True)
    tolerance = np.finfo(float).eps * np.maximum(np.max(np.abs(data), axis=1, keepdims=True), np.finfo(float).tiny)
    normalized = np.zeros_like(data)
    np.divide(centered, 1.4826 * mad, out=normalized, where=mad > tolerance)
    return normalized.astype(np.float32)

data = preprocess(raw_data, sampling_rate, PARAMETERS)
# 暂时保留 raw_data，供下方 preprocessv2 与原版对比。
print(f'跟踪数据：{data.shape[0]} 个通道，{data.shape[1]} 个原始时间采样点')
# 260000是因为 60秒 * 520个通道 * 1000Hz采样率 + 前后补充了100秒


In [ ]:
from daspy.basic_tools.filter import bandpass as daspy_bandpass
from daspy.advanced_tools.denoising import common_mode_noise_removal
from daspy.basic_tools.preprocessing import normalization as daspy_normalization
from daspy.basic_tools.preprocessing import detrending


def preprocessv2(raw_data, sampling_rate, parameters, *, on_stage=None):
    """复用 DASPy；on_stage 可接收所选通道的各阶段结果。"""
    channel_from, channel_to = parameters['channel_from'], parameters['channel_to']
    channels = slice(channel_from - 1, channel_to)

    def record(name, values):
        if on_stage is not None:
            on_stage(name, values)

    record('Raw data', raw_data[channels])
    
    # 1. 去趋势 - 去除长时间漂移
    detrended = np.empty_like(raw_data, dtype=np.float32)
    for i in range(len(raw_data)):
        detrended[i] = detrending(raw_data[i])
    record('Detrended', detrended[channels])
    
    # 2. 带通滤波
    filtered = np.empty_like(detrended, dtype=np.float32)
    for first in range(0, len(detrended), 8):
        filtered[first:first + 8] = daspy_bandpass(
            detrended[first:first + 8], fs=sampling_rate,
            freqmin=parameters['frequency_low'], freqmax=parameters['frequency_high'],
            corners=4, zerophase=True)
    record('Butterworth bandpass', filtered[channels])

    # 3. 共模噪声去除 - DASPy 未检查公共波形能量为零，因此保留原版保护。
    common = np.median(filtered, axis=0)
    if float(np.sum(common ** 2)) > np.finfo(float).eps:
        filtered = common_mode_noise_removal(filtered, method='median')
    record('Common-mode noise removed', filtered[channels])

    # 4. 通道选择与MAD归一化
    selected = filtered[channels].astype(float)
    del filtered
    centered = selected - np.median(selected, axis=1, keepdims=True)
    record('Channel median removed', centered)
    mad = np.median(np.abs(centered), axis=1, keepdims=True)
    tolerance = np.finfo(float).eps * np.maximum(
        np.max(np.abs(selected), axis=1, keepdims=True), np.finfo(float).tiny)
    valid = (mad > tolerance).ravel()
    normalized = np.zeros_like(selected)
    if np.any(valid):
        # DASPy 的 MAD 分母不含 1.4826；补上以保持原幅度尺度。
        normalized[valid] = daspy_normalization(selected[valid], method='MAD') / 1.4826
    result = normalized.astype(np.float32)
    record('MAD normalized (scale 1.4826)', result)
    return result


In [ ]:
from importlib.metadata import version

PLOT_CHANNEL = (PARAMETERS['channel_from'] + PARAMETERS['channel_to']) // 2 #绘图选择哪个通道
preview_first = max(0, round(WINDOW_START * sampling_rate) - sample_origin) #这个是绘图的要选择在哪里开始看
preview_last = min(raw_data.shape[1], round(WINDOW_END * sampling_rate) - sample_origin) #这个是回退的要看的结尾
trace_times = (np.arange(preview_first, preview_last) + sample_origin) / sampling_rate #给出来时间轴
trace_row = PLOT_CHANNEL - PARAMETERS['channel_from'] #这个是选中的通道在数据中的行号
preprocess_stages = {}


def capture_preprocess_stage(name, values):
    # 保存目标窗口的全部采样点，用于绘图。
    window = values[:, preview_first:preview_last]
    preprocess_stages[name] = dict(
        image=window.astype(np.float32, copy=True),
        trace=window[trace_row].astype(np.float32, copy=True))


data_v1 = data #原版本
data_v2 = preprocessv2(raw_data, sampling_rate, PARAMETERS, on_stage=capture_preprocess_stage)
data = data_v2  # 后续找峰、跟踪使用新版结果。
del raw_data
print(f'DASPy {version("DASPy-toolbox")}；输出形状 {data_v2.shape}')
extent = [trace_times[0] - 0.5 / sampling_rate,
          trace_times[-1] + 0.5 / sampling_rate,
          PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5]

fig, axes = plt.subplots(len(preprocess_stages), 2, figsize=(27, 25),
                         sharex=True, layout='constrained')
for row, (name, stage) in enumerate(preprocess_stages.items()):
    image_ax, trace_ax = axes[row]
    limit = max(float(np.percentile(np.abs(stage['image']), 99)), 1e-8)
    artist = image_ax.imshow(stage['image'], origin='lower', aspect='auto', cmap='RdBu_r',
                             interpolation='none', extent=extent, vmin=-limit, vmax=limit)
    label = 'Normalized amplitude' if name.startswith('MAD') else 'Amplitude'
    fig.colorbar(artist, ax=image_ax, label=label, fraction=0.035, pad=0.02)
    image_ax.set(title=f'{row + 1}. {name}', ylabel='Channel')
    trace_ax.plot(trace_times, stage['trace'], linewidth=0.7)
    trace_ax.axhline(0, color='0.5', linewidth=0.5)
    trace_ax.set(title=f'Channel {PLOT_CHANNEL}', ylabel=label)
    trace_ax.grid(alpha=0.25)
    for ax in (image_ax, trace_ax):
        ax.set_xlim(trace_times[0], trace_times[-1] + 1 / sampling_rate)
for ax in axes[-1]:
    ax.set_xlabel('Time from selected file start [s]')
plt.show()

In [ ]:
# 找峰参数统一在第一个参数单元设置，避免后面的单元覆盖参数。
def find_channel_peaks(data, sampling_rate, parameters, polarity):
    peaks, strengths = [], [] # 第一个参数是每个通道的峰位置，第二个参数是每个通道的峰的强度
    for trace in data: #遍历每一个通道
        locations, properties = signal.find_peaks( #开始找峰值
            polarity * trace, prominence=parameters['peak_prominence'], #第一个参数是峰相对于周围基底至少突出多少
            distance=round(parameters['minimum_separation'] * sampling_rate), #保留的峰至少间隔多少
            wlen=round(parameters['prominence_window'] * sampling_rate) | 1) #突出程度取多少窗口计算
        peaks.append(locations)
        strengths.append(properties['prominences']) ##第一个参数是峰相对于周围基底至少突出多少
    return peaks, strengths


polarities = (1,)
detected_peaks_prominence = {
    polarity: find_channel_peaks(data, sampling_rate, PARAMETERS, polarity)
    for polarity in polarities
}
first = round(WINDOW_START * sampling_rate) - sample_origin
last = round(WINDOW_END * sampling_rate) - sample_origin
visible_data = data[:, first:last]
from daspy.basic_tools.visualization import plot as daspy_plot

figure_dir = Path('figures')
figure_dir.mkdir(exist_ok=True)


In [ ]:
# 局部霍夫曲线：支持弱峰、相接片段连接和通道密度筛选。
"""局部霍夫找种子，再用低阶曲线连接各通道的峰点。"""

import numpy as np
from numpy.polynomial import Polynomial
from skimage.transform import hough_line, hough_line_peaks


HOUGH_PARAMETERS = dict(
    time_step=0.05,                 # 霍夫图的时间格宽度，单位 s。
    angle_bins=1800,
    window_channels=60,             # 每次只在局部通道窗口内找直线。
    window_overlap=30,
    vote_threshold=12,              # 局部窗口的票数门槛。
    minimum_seed_channels=12,       # 初始轨迹至少需要的支持通道数。
    minimum_track_density=0.5,      # 支持通道数 / 轨迹覆盖的通道数。
    extension_channels=15,          # 常规搜索步长；至少覆盖允许的连续缺失长度。
    match_tolerance=0.5,           # 原始峰与预测到达时间的最大偏差，单位 s。
    curve_min_improvement=0.015,    # 二次拟合至少降低多少 RMSE 才采用，单位 s。
    duplicate_tolerance=0.5,       # 重复轨迹的预测时间容差，单位 s。
    duplicate_fraction=0.6,         # 重叠范围与共同支持通道上的一致比例。
    merge_slope_tolerance=0.25,     # 相接片段在连接处允许的相对斜率差。
)


def collect_peaks(channel_peaks, sampling_rate, sample_origin,
                  window_start, window_end):
    """保留原始峰时刻，同时记录每个通道对应的峰索引。"""
    peak_times = []
    peak_rows = []
    indices_by_row = []
    for row, locations in enumerate(channel_peaks):
        times = (np.asarray(locations) + sample_origin) / sampling_rate
        times = np.sort(times[(times >= window_start) & (times < window_end)])
        first_index = len(peak_times)
        indices_by_row.append(np.arange(first_index, first_index + len(times)))
        peak_times.extend(times)
        peak_rows.extend([row] * len(times))
    return np.asarray(peak_times), np.asarray(peak_rows, dtype=int), indices_by_row


def match_peaks(peak_times, indices_by_row, model, first_row, last_row, tolerance):
    """在闭区间 [first_row, last_row] 内，每个通道选最近的一个峰。"""
    selected = []
    for row in range(first_row, last_row + 1):
        candidates = indices_by_row[row]
        if len(candidates) == 0:
            continue
        errors = np.abs(peak_times[candidates] - model(row))
        nearest = np.argmin(errors)
        if errors[nearest] <= tolerance:
            selected.append(candidates[nearest])
    return np.asarray(selected, dtype=int)


def connected_group(selected, peak_rows, maximum_missed_channels, reference=None):
    """遇到长空白就分段；延伸时优先保留与已有轨迹重合最多的一段。"""
    if len(selected) == 0:
        return selected
    gaps = np.flatnonzero(
        np.diff(peak_rows[selected]) > maximum_missed_channels + 1
    ) + 1
    groups = np.split(selected, gaps)
    best_group = groups[0]
    best_score = (-1, -1)
    for group in groups:
        overlap = 0
        if reference is not None:
            overlap = len(np.intersect1d(group, reference))
        score = (overlap, len(group))
        if score > best_score:
            best_group = group
            best_score = score
    return best_group


def fit_curve(rows, times, settings):
    """优先使用直线；二次曲线明显改善误差时才增加模型复杂度。"""
    line = Polynomial.fit(rows, times, deg=1)
    curve = Polynomial.fit(rows, times, deg=2)
    line_rmse = np.sqrt(np.mean((times - line(rows)) ** 2))
    curve_rmse = np.sqrt(np.mean((times - curve(rows)) ** 2))
    if line_rmse - curve_rmse >= settings['curve_min_improvement']:
        return curve
    return line


def valid_motion(model, first_row, last_row, parameters):
    """整个轨迹范围内方向一致，且局部速度都满足限制。"""
    rows = np.arange(first_row, last_row + 1)
    slopes = model.deriv()(rows)
    if np.all(slopes > 0):
        direction = 1
    elif np.all(slopes < 0):
        direction = -1
    else:
        return False
    speed = parameters['channel_spacing'] / np.abs(slopes)
    if np.any(speed < parameters['minimum_speed']):
        return False
    if np.any(speed > parameters['maximum_speed']):
        return False
    if parameters['direction'] == 'increasing' and direction < 0:
        return False
    if parameters['direction'] == 'decreasing' and direction > 0:
        return False
    return True


def fit_track(selected, peak_times, peak_rows, parameters, settings, votes):
    """拟合后剔除超出时间容差的点，保证输出点与最终模型一致。"""
    minimum = settings['minimum_seed_channels']
    while len(selected) >= minimum:
        rows = peak_rows[selected]
        times = peak_times[selected]
        model = fit_curve(rows, times, settings)
        errors = np.abs(times - model(rows))
        accepted = selected[errors <= settings['match_tolerance']]
        accepted = connected_group(
            accepted, peak_rows, parameters['maximum_missed_channels'], selected
        )
        if np.array_equal(accepted, selected):
            if not valid_motion(model, rows[0], rows[-1], parameters):
                return None
            return dict(selected=selected, model=model, votes=votes,
                        rmse=float(np.sqrt(np.mean(errors ** 2))))
        selected = accepted
    return None


def grow_track(seed, peak_times, peak_rows, indices_by_row, parameters, settings, votes):
    """逐段延伸，每轮都用原始峰时刻重新拟合，避免长距离直接外推。"""
    track = fit_track(seed, peak_times, peak_rows, parameters, settings, votes)
    if track is None:
        return None
    num_channels = len(indices_by_row)
    # 搜索范围必须能到达空白段后的首个通道，否则会在允许的缺口前提前停止。
    step = max(settings['extension_channels'], parameters['maximum_missed_channels'] + 1)
    # 足以从任意种子延伸到边界，并留几轮做峰点重匹配。
    iterations = int(np.ceil(num_channels / step)) + 4
    for _ in range(iterations):
        selected = track['selected']
        first_row = max(0, peak_rows[selected[0]] - step)
        last_row = min(num_channels - 1, peak_rows[selected[-1]] + step)
        matched = match_peaks(
            peak_times, indices_by_row, track['model'], first_row, last_row,
            settings['match_tolerance']
        )
        matched = connected_group(
            matched, peak_rows, parameters['maximum_missed_channels'], selected
        )
        updated = fit_track(matched, peak_times, peak_rows, parameters, settings, votes)
        if updated is None:
            break
        track = updated
        if np.array_equal(track['selected'], selected):
            break
    return track


def find_seeds(peak_image, peak_times, peak_rows, indices_by_row,
               window_start, parameters, settings):
    """在重叠的通道窗口里做直线霍夫，输出原始峰点组成的局部种子。"""
    num_channels = len(indices_by_row)
    width = settings['window_channels']
    step = width - settings['window_overlap']
    time_step = settings['time_step']
    theta = np.linspace(-np.pi / 2, np.pi / 2, settings['angle_bins'], endpoint=False)
    seeds = []
    local_hough = []
    for first_row in range(0, num_channels, step):
        end_row = min(first_row + width, num_channels)
        if end_row - first_row < settings['minimum_seed_channels']:
            break
        image = peak_image[first_row:end_row]
        accumulator, angles, distances = hough_line(image, theta=theta)
        votes, candidate_angles, candidate_distances = hough_line_peaks(
            accumulator, angles, distances, threshold=settings['vote_threshold'],
            min_distance=3, min_angle=5
        )
        local_hough.append(dict(
            first_row=first_row, end_row=end_row, accumulator=accumulator,
            angles=angles, distances=distances, candidate_count=len(votes)
        ))
        for vote, angle, rho in zip(votes, candidate_angles, candidate_distances):
            if np.isclose(np.cos(angle), 0):
                continue
            slope = -np.tan(angle) * time_step
            # 霍夫图的 y 从窗口首通道开始，转为全局相对通道号时需减去偏移。
            intercept = window_start + (rho / np.cos(angle) + 0.5) * time_step
            intercept -= slope * first_row
            model = Polynomial([intercept, slope])
            if not valid_motion(model, first_row, end_row - 1, parameters):
                continue
            tolerance = settings['match_tolerance']
            tolerance += 0.5 * time_step / abs(np.cos(angle))
            selected = match_peaks(
                peak_times, indices_by_row, model, first_row, end_row - 1, tolerance
            )
            selected = connected_group(
                selected, peak_rows, parameters['maximum_missed_channels']
            )
            if len(selected) >= settings['minimum_seed_channels']:
                seeds.append((selected, int(vote)))
    return seeds, local_hough


def track_quality(track, tolerance):
    """综合支持点数与误差，避免多几个点的混合轨迹压过拟合准确的轨迹。"""
    relative_error = track['rmse'] / tolerance
    return len(track['selected']) / (1 + relative_error ** 2)


def adjoining_segments(first, second, peak_rows, parameters, settings):
    """共同通道很少时，按端点时间与局部斜率判断是否为相接片段。"""
    first_rows = peak_rows[first['selected']]
    second_rows = peak_rows[second['selected']]
    if first_rows[0] > second_rows[0]:
        first, second = second, first
        first_rows, second_rows = second_rows, first_rows
    gap = second_rows[0] - first_rows[-1] - 1
    if gap > parameters['maximum_missed_channels']:
        return False
    if gap < -settings['minimum_seed_channels']:
        return False
    join_row = (first_rows[-1] + second_rows[0]) / 2
    difference = abs(first['model'](join_row) - second['model'](join_row))
    if difference > settings['duplicate_tolerance']:
        return False
    first_slope = first['model'].deriv()(join_row)
    second_slope = second['model'].deriv()(join_row)
    if first_slope * second_slope <= 0:
        return False
    relative_difference = abs(first_slope - second_slope) / max(abs(first_slope), abs(second_slope))
    return relative_difference <= settings['merge_slope_tolerance']


def merge_pair(first, second, peak_times, peak_rows, parameters, settings):
    """合并重叠重复候选，或时间、斜率连续的相接片段。"""
    first_rows = peak_rows[first['selected']]
    second_rows = peak_rows[second['selected']]
    common_rows = np.intersect1d(first_rows, second_rows)
    same_observations = False
    if len(common_rows) < settings['minimum_seed_channels']:
        if not adjoining_segments(first, second, peak_rows, parameters, settings):
            return None
    else:
        overlap = min(first_rows[-1], second_rows[-1]) - max(first_rows[0], second_rows[0]) + 1
        shorter_span = min(first_rows[-1] - first_rows[0] + 1,
                           second_rows[-1] - second_rows[0] + 1)
        if overlap < settings['duplicate_fraction'] * shorter_span:
            return None
        first_slopes = first['model'].deriv()(common_rows)
        second_slopes = second['model'].deriv()(common_rows)
        if np.any(first_slopes * second_slopes <= 0):
            return None
        # 按共同有观测的通道比较，避免因两边漏检位置不同低估重合比例。
        minimum_shared = settings['duplicate_fraction'] * len(common_rows)
        shared_points = len(np.intersect1d(first['selected'], second['selected']))
        same_observations = shared_points >= minimum_shared
        differences = np.abs(first['model'](common_rows) - second['model'](common_rows))
        close_predictions = np.count_nonzero(differences <= settings['duplicate_tolerance'])
        if not same_observations and close_predictions < minimum_shared:
            return None
    # 每通道仍只保留一个峰；重叠部分优先采用排序在前的较优轨迹。
    new_points = second['selected'][~np.isin(second_rows, first_rows)]
    selected = np.concatenate((first['selected'], new_points))
    selected = selected[np.argsort(peak_rows[selected])]
    model = fit_curve(peak_rows[selected], peak_times[selected], settings)
    # 检查两边都有足够多的点支持联合模型；少量边缘点超差不应阻止去重。
    for track in [first, second]:
        indices = track['selected']
        errors = np.abs(peak_times[indices] - model(peak_rows[indices]))
        supported = np.count_nonzero(errors <= settings['match_tolerance'])
        if supported < settings['duplicate_fraction'] * len(indices):
            if same_observations:
                return first
            return None
    votes = max(first['votes'], second['votes'])
    merged = fit_track(selected, peak_times, peak_rows, parameters, settings, votes)
    if merged is not None and len(merged['selected']) >= len(first['selected']):
        return merged
    # 共享大部分原始峰时已能确认重复；联合拟合无改善时保留较优轨迹。
    if same_observations:
        return first
    return None


def merge_tracks(proposals, peak_times, peak_rows, parameters, settings):
    """先处理支持多、误差小的轨迹；合并后再扫描，处理可连接的重复片段。"""
    tracks = proposals
    previous_count = -1
    while len(tracks) != previous_count:
        previous_count = len(tracks)
        tracks.sort(key=lambda track: track_quality(track, settings['match_tolerance']), reverse=True)
        retained = []
        for track in tracks:
            merged = False
            for index, other in enumerate(retained):
                combined = merge_pair(other, track, peak_times, peak_rows, parameters, settings)
                if combined is not None:
                    retained[index] = combined
                    merged = True
                    break
            if not merged:
                retained.append(track)
        tracks = retained
    return tracks


def pick_hough_tracks(channel_peaks, sampling_rate, sample_origin,
                      window_start, window_end, parameters, settings=None):
    """局部霍夫种子 → 曲线延伸 → 联合拟合去重；速度单位始终为 m/s。

    channel_peaks 中的位置是相对 sample_origin 的采样点索引。
    每个通道最多选一个峰；轨迹模型为 t = model(相对通道号)。
    settings 可只传需要修改的项目，其余使用 HOUGH_PARAMETERS。
    """
    options = HOUGH_PARAMETERS.copy()
    if settings is not None:
        options.update(settings)
    times, rows, indices_by_row = collect_peaks(
        channel_peaks, sampling_rate, sample_origin, window_start, window_end
    )
    num_bins = int(np.ceil((window_end - window_start) / options['time_step']))
    peak_image = np.zeros((len(channel_peaks), num_bins), dtype=bool)
    time_bins = np.floor((times - window_start) / options['time_step']).astype(int)
    peak_image[rows, time_bins] = True
    seeds, local_hough = find_seeds(
        peak_image, times, rows, indices_by_row, window_start, parameters, options
    )
    proposals = []
    for selected, votes in seeds:
        track = grow_track(
            selected, times, rows, indices_by_row, parameters, options, votes
        )
        if track is not None:
            proposals.append(track)
    tracks = merge_tracks(proposals, times, rows, parameters, options)
    results = []
    for track in tracks:
        selected = track['selected']
        if len(selected) < parameters['minimum_track_channels']:
            continue
        support_fraction = len(selected) / (rows[selected[-1]] - rows[selected[0]] + 1)
        if support_fraction < options['minimum_track_density']:
            continue
        model = track['model']
        slopes = model.deriv()(rows[selected])
        speeds = parameters['channel_spacing'] / np.abs(slopes)
        samples = np.rint(times[selected] * sampling_rate).astype(np.int64)
        results.append(dict(
            points=np.column_stack((samples, rows[selected] + parameters['channel_from'])),
            rows=rows[selected], model=model, support_fraction=float(support_fraction),
            speed=float(np.median(speeds)), speed_profile=speeds,
            direction=1 if slopes[0] > 0 else -1,
            time_per_channel=slopes, time_at_first_channel=float(model(0)),
            votes=track['votes'], rmse=track['rmse']
        ))
    results.sort(key=lambda track: track['points'][:, 0].min())
    diagnostics = dict(
        peak_image=peak_image, local_hough=local_hough,
        candidate_count=sum(item['candidate_count'] for item in local_hough),
        seed_count=len(seeds), proposal_count=len(proposals)
    )
    return results, diagnostics


In [ ]:
"""二维脊线验证一维候选峰；输出位置始终是原始波形的局部极大值。"""

import numpy as np
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from skimage.feature import hessian_matrix, hessian_matrix_eigvals


RIDGE_PARAMETERS = dict(
    time_step=None,                 # 默认不降采样；显式给出秒数才允许降采样。
    scale_time_step=0.1,            # 尺度的时间单位（秒），与计算网格分辨率分开。
    scales=(1.0, 2.0, 3.0),         # 通道 sigma=scale；时间 sigma=scale*0.1 秒。
    hessian_backend='auto',         # 长时间核使用 FFT；skimage 可用于数值对照。
    padding_seconds=5.0,
    candidate_prominence=1.5,       # 宽松候选池，弱峰先保留。
    minimum_width=0.15,             # s；排除很窄的毛刺。
    ridge_threshold=0.02,           # 相对于图中高响应区域的脊线强度。
    neighbor_channels=6,            # 向上下各检查 6 个通道。
    strong_neighbors=2,             # 原本达到突出度门槛的峰，至少需要 2 个邻道支持。
    weak_neighbors=3,               # 补入的弱峰，需要更严格的邻道支持。
    neighbor_tolerance=0.35,        # s；邻道峰偏离局部方向预测的最大时间差。
)

#这里计算的是海森矩阵
def ridge_hessian(image, sigma, backend='auto'):
    """保持两次高斯一阶导数及零填充边界，仅加速长时间核的卷积。"""
    if backend not in ('auto', 'fft', 'skimage'):
        raise ValueError('hessian_backend 必须为 auto、fft 或 skimage')
    # 与 skimage 的 Gaussian-derivative Hessian 使用相同的核宽和截断。
    truncate = 8 if all(s > 1 for s in sigma) else 100
    row_sigma, time_sigma = (s / np.sqrt(2) for s in sigma)
    radius = int(truncate * time_sigma + 0.5)
    if backend == 'skimage' or (backend == 'auto' and (radius < 64 or image.shape[1] < 256)):
        return hessian_matrix(
            image, sigma=sigma, order='rc', use_gaussian_derivatives=True
        )

    positions = np.arange(-radius, radius + 1, dtype=float)
    gaussian = np.exp(-0.5 * (positions / time_sigma) ** 2)
    gaussian /= gaussian.sum()
    kernels = (gaussian, -positions / time_sigma ** 2 * gaussian)

    def derivative(values, row_order, time_order):
        filtered = gaussian_filter1d(
            values, row_sigma, axis=0, order=row_order,
            mode='constant', cval=0, truncate=truncate,
        )
        # 分批计算，限制 FFT 临时数组的大小；每批仍保留所有时间点。
        for first_row in range(0, len(filtered), 16):
            block = filtered[first_row:first_row + 16]
            block[:] = signal.fftconvolve(
                block, kernels[time_order][None, :], mode='same', axes=1
            )
        return filtered

    row_gradient = derivative(image, 1, 0)
    time_gradient = derivative(image, 0, 1)
    hrr = derivative(row_gradient, 1, 0)
    hrt = derivative(row_gradient, 0, 1)
    del row_gradient
    htt = derivative(time_gradient, 0, 1)
    return hrr, hrt, htt


def ridge_response(image, settings, time_step=0.1):
    """保留每个时间点，用固定物理尺度计算并归一化 Hessian。"""
    response = np.zeros(image.shape, dtype=float)
    # 在“1 通道 × scale_time_step 秒”的参考坐标中比较曲率。
    # 时间网格变细时增大时间 sigma，并补偿二阶导数的坐标缩放。
    time_ratio = settings.get('scale_time_step', 0.1) / time_step
    for scale in settings['scales']:
        hrr, hrt, htt = ridge_hessian(
            image, sigma=(scale, scale * time_ratio),
            backend=settings.get('hessian_backend', 'auto'),
        )
        hrt *= time_ratio
        htt *= time_ratio ** 2
        # 特征值按数值从大到小排列；亮脊线的横向曲率为较大的负值。
        along, across = hessian_matrix_eigvals([hrr, hrt, htt])
        curvature = np.maximum(-across, 0)
        ratio = np.abs(along) / np.maximum(curvature, np.finfo(float).eps)
        strength = scale ** 2 * curvature * np.exp(-0.5 * (ratio / 0.5) ** 2)

        response = np.maximum(response, strength)
    return response


def local_slopes(duration, parameters, settings):
    """短邻域内验证一组可能方向，避免噪声使单点方向估计失准。"""
    minimum = parameters['channel_spacing'] / parameters['maximum_speed']
    # 在窗口内跨越最小轨迹长度时的平均时间斜率，给局部搜索一个上界。
    maximum = duration / (parameters['minimum_track_channels'] - 1)
    if parameters['minimum_speed'] > 0:
        maximum = min(maximum, parameters['channel_spacing'] / parameters['minimum_speed'])
    maximum = max(minimum, maximum)
    step = settings['neighbor_tolerance'] / settings['neighbor_channels']
    positive = np.arange(minimum, maximum + step / 2, step)
    if parameters['direction'] == 'increasing':
        return positive
    if parameters['direction'] == 'decreasing':
        return -positive
    return np.concatenate((-positive, positive))


def nearby_peak_mask(locations, expected, tolerance):
    """判断每个预测位置附近，是否存在另一个通道的候选峰。"""
    observed = nearest_peak(locations, expected)
    return np.abs(observed - expected) <= tolerance


def nearest_peak(locations, expected):
    """返回最近的邻道峰；空通道用 NaN 表示无观测。"""
    if len(locations) == 0:
        return np.full(expected.shape, np.nan)
    indices = np.searchsorted(locations, expected)
    left = locations[np.maximum(indices - 1, 0)]
    right = locations[np.minimum(indices, len(locations) - 1)]
    return np.where(np.abs(left - expected) <= np.abs(right - expected), left, right)


def filter_ridge_peaks_by_tracks(detections, diagnostics, tracks,
                                 sample_origin, channel_from):
    """只保留已验证长轨迹实际使用的原始峰，不沿模型生成或补齐峰点。

    tracks 来自 pick_hough_tracks，points 的时间坐标为含 sample_origin 的采样点。
    返回接口与 find_channel_peaks_ridge 相同；局部检测结果由调用者另存以便审查。
    """
    peaks, strengths = detections
    supported = [[] for _ in peaks]
    for track in tracks:
        for sample, channel in track['points']:
            row = int(channel) - channel_from
            if 0 <= row < len(peaks):
                supported[row].append(int(sample) - sample_origin)
    masks = [np.isin(locations, observed) for locations, observed in zip(peaks, supported)]
    retained = [locations[mask] for locations, mask in zip(peaks, masks)]
    retained_strengths = [values[mask] for values, mask in zip(strengths, masks)]
    updated = diagnostics.copy()
    for key in ('ridge_scores', 'neighbor_counts'):
        updated[key] = [values[mask] for values, mask in zip(diagnostics[key], masks)]
    updated['local_selected_count'] = sum(map(len, peaks))
    updated['selected_count'] = sum(map(len, retained))
    updated['track_rejected_count'] = updated['local_selected_count'] - updated['selected_count']
    updated['track_count'] = len(tracks)
    return (retained, retained_strengths), updated


def find_channel_peaks_ridge(data, sampling_rate, parameters, polarity,
                             first, last, settings=None):
    """返回 ((各通道原始峰索引, prominence), 诊断数据)。

    first、last 是 data 中的窗口索引，区间为 [first, last)。
    二维图只判断候选峰是否有线状结构支持，不用于生成或移动峰位置。
    """
    options = RIDGE_PARAMETERS.copy()
    if settings is not None:
        options.update(settings)
    padding = round(options['padding_seconds'] * sampling_rate)
    left = max(0, first - padding)
    right = min(data.shape[1], last + padding)
    requested_step = options['time_step']
    if requested_step is not None and (not np.isfinite(requested_step) or requested_step <= 0):
        raise ValueError('time_step 必须为 None（不降采样）或正的有限秒数')
    stride = 1 if requested_step is None else max(1, round(requested_step * sampling_rate))
    time_step = stride / sampling_rate
    image = polarity * np.asarray(data[:, left:right], dtype=float)
    if stride > 1:
        image = signal.resample_poly(image, 1, stride, axis=1)
    response = ridge_response(image, options, time_step)
    grid_samples = left + np.arange(image.shape[1]) * stride
    visible = (grid_samples >= first) & (grid_samples < last)
    reference = np.percentile(response[:, visible], 99.5)
    if reference > 0:
        response = np.clip(response / reference, 0, 1)

    candidates = []
    anchors = []
    prominences = []
    candidate_scores = []
    raw_candidate_count = 0
    for row, trace in enumerate(data[:, left:right]):
        reference_peaks, _ = signal.find_peaks(
            polarity * trace,
            prominence=parameters['peak_prominence'],
            distance=round(parameters['minimum_separation'] * sampling_rate),
            wlen=round(parameters['prominence_window'] * sampling_rate) | 1,
        )
        reference_peaks += left
        reference_columns = np.rint((reference_peaks - left) / stride).astype(int)
        reference_columns = np.minimum(reference_columns, image.shape[1] - 1)
        reference_valid = response[row, reference_columns] >= options['ridge_threshold']
        reference_valid &= polarity * data[row, reference_peaks] > 0
        anchors.append(reference_peaks[reference_valid])
        locations, properties = signal.find_peaks(
            polarity * trace,
            prominence=options['candidate_prominence'],
            distance=round(parameters['minimum_separation'] * sampling_rate),
            width=options['minimum_width'] * sampling_rate,
            wlen=round(parameters['prominence_window'] * sampling_rate) | 1,
        )
        locations += left
        raw_candidate_count += np.count_nonzero((locations >= first) & (locations < last))
        columns = np.rint((locations - left) / stride).astype(int)
        columns = np.minimum(columns, image.shape[1] - 1)
        scores = response[row, columns]
        eligible = (scores >= options['ridge_threshold']) & (polarity * data[row, locations] > 0)
        candidates.append(locations[eligible])
        prominences.append(properties['prominences'][eligible])
        candidate_scores.append(scores[eligible])

    peaks, strengths = [], []
    ridge_scores, neighbor_counts = [], []
    radius = options['neighbor_channels']
    tolerance = options['neighbor_tolerance'] * sampling_rate
    slopes = local_slopes((last - first) / sampling_rate, parameters, options)
    for row, locations in enumerate(candidates):
        support = np.zeros((len(locations), len(slopes)), dtype=int)
        first_row = max(0, row - radius)
        last_row = min(len(candidates), row + radius + 1)
        for neighbor in range(first_row, last_row):
            if neighbor == row:
                continue
            expected = locations[:, None] + slopes[None, :] * (neighbor - row) * sampling_rate
            support += nearby_peak_mask(anchors[neighbor], expected, tolerance)
        best_direction = np.argmax(support, axis=1)
        best_slopes = slopes[best_direction]
        support = np.max(support, axis=1)
        # 用实际匹配峰再检查斜率，避免同一时刻的公共干扰因容差而冒充运动轨迹。
        numerator = np.zeros(len(locations))
        denominator = np.zeros(len(locations))
        for neighbor in range(first_row, last_row):
            if neighbor == row:
                continue
            offset = neighbor - row
            expected = locations + best_slopes * offset * sampling_rate
            observed = nearest_peak(anchors[neighbor], expected)
            matched = np.abs(observed - expected) <= tolerance
            numerator[matched] += offset * (observed[matched] - locations[matched])
            denominator[matched] += offset ** 2
        observed_slopes = np.zeros(len(locations))
        np.divide(numerator, denominator * sampling_rate, out=observed_slopes,
                  where=denominator > 0)
        minimum_slope = parameters['channel_spacing'] / parameters['maximum_speed']
        # 短邻域估计有噪声，此处只排除近零斜率；完整速度范围在轨迹阶段检查。
        valid_motion = np.abs(observed_slopes) >= 0.5 * minimum_slope
        valid_motion &= observed_slopes * best_slopes > 0
        if parameters['minimum_speed'] > 0:
            valid_motion &= np.abs(observed_slopes) <= parameters['channel_spacing'] / parameters['minimum_speed']
        strong = prominences[row] >= parameters['peak_prominence']
        required = np.where(strong, options['strong_neighbors'], options['weak_neighbors'])
        selected = (locations >= first) & (locations < last)
        selected &= support >= required
        selected &= valid_motion
        peaks.append(locations[selected])
        strengths.append(prominences[row][selected])
        ridge_scores.append(candidate_scores[row][selected])
        neighbor_counts.append(support[selected])

    diagnostics = dict(
        response=response[:, visible].astype(np.float32),
        grid_samples=grid_samples[visible], ridge_scores=ridge_scores,
        neighbor_counts=neighbor_counts, raw_candidate_count=raw_candidate_count,
        selected_count=sum(len(locations) for locations in peaks),
        time_step=time_step,
    )
    return (peaks, strengths), diagnostics


# 局部响应筛选后，再用长轨迹约束；保留全部局部结果供对照。
detected_peaks_ridge_local, ridge_local_diagnostics = {}, {}
detected_peaks_ridge, ridge_diagnostics = {}, {}
ridge_track_candidates, ridge_track_diagnostics = {}, {}
print(f"长轨迹约束：至少 {PARAMETERS['minimum_track_channels']} 个支持通道，"
      f"覆盖率 >= {HOUGH_PARAMETERS['minimum_track_density']:.0%}，"
      f"时间容差 {HOUGH_PARAMETERS['match_tolerance']:g} s")
for polarity in polarities:
    local_detections, local_diagnostics = find_channel_peaks_ridge(
        data, sampling_rate, PARAMETERS, polarity, first, last, RIDGE_PARAMETERS
    )
    tracks, track_diagnostics = pick_hough_tracks(
        local_detections[0], sampling_rate, sample_origin,
        WINDOW_START, WINDOW_END, PARAMETERS, HOUGH_PARAMETERS
    )
    for track in tracks:
        track['polarity'] = polarity
    detections, diagnostics = filter_ridge_peaks_by_tracks(
        local_detections, local_diagnostics, tracks, sample_origin,
        PARAMETERS['channel_from']
    )
    detected_peaks_ridge_local[polarity] = local_detections
    ridge_local_diagnostics[polarity] = local_diagnostics
    detected_peaks_ridge[polarity] = detections
    ridge_diagnostics[polarity] = diagnostics
    ridge_track_candidates[polarity] = tracks
    ridge_track_diagnostics[polarity] = track_diagnostics
    print(f"极性 {polarity:+d}：宽松候选 {diagnostics['raw_candidate_count']}，"
          f"局部通过 {diagnostics['local_selected_count']}，"
          f"长轨迹筛选后 {diagnostics['selected_count']} 个原始峰；"
          f"剔除 {diagnostics['track_rejected_count']} 个点，{len(tracks)} 条轨迹")
detected_peaks = detected_peaks_ridge

# 上图为波形，中图为二维脊线强度，下图检查选中点是否贴着原始波形。
extent = [WINDOW_START - 0.5 / sampling_rate, WINDOW_END - 0.5 / sampling_rate,
          PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5]
limit = np.percentile(np.abs(visible_data), 99)
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True, sharey=True,
                         layout='constrained')
for ax in [axes[0], axes[2]]:
    ax.imshow(visible_data, origin='lower', aspect='auto', cmap='RdBu_r',
              extent=extent, vmin=-limit, vmax=limit, interpolation='none')
axes[0].set_title('Preprocessed signal')
first_polarity = polarities[0]
diagnostics = ridge_diagnostics[first_polarity]
grid_times = (diagnostics['grid_samples'] + sample_origin) / sampling_rate
grid_extent = [grid_times[0] - diagnostics['time_step'] / 2,
               grid_times[-1] + diagnostics['time_step'] / 2,
               PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5]
artist = axes[1].imshow(
    diagnostics['response'], origin='lower', aspect='auto', cmap='magma',
    extent=grid_extent, vmin=0, vmax=1, interpolation='none'
)
fig.colorbar(artist, ax=axes[1], label='Relative ridge response', fraction=0.025)
axes[1].set_title(f'2D ridge response: polarity {first_polarity:+d}')
for polarity, (channel_peaks, _) in detected_peaks_ridge.items():
    peak_times, peak_channels = [], []
    for row, locations in enumerate(channel_peaks):
        peak_times.extend((locations + sample_origin) / sampling_rate)
        peak_channels.extend([PARAMETERS['channel_from'] + row] * len(locations))
    color = '#1f77b4' if polarity > 0 else '#2ca02c'
    axes[2].scatter(peak_times, peak_channels, s=5, c=color,
                    label=f'Polarity {polarity:+d}: {len(peak_times)} peaks')
axes[2].set_title('Ridge + long-track validation: original extrema only')
axes[2].legend(loc='upper right')
for ax in axes:
    ax.set(ylabel='Channel', xlim=(WINDOW_START, WINDOW_END))
axes[2].set_xlabel('Time from selected file start [s]')
fig.savefig(figure_dir / 'ridge_peak_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

# 独立对照图显示局部筛选与长轨迹筛选，底图和色标完全一致。
fig, comparison_axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True, sharey=True,
                                    layout='constrained')
for ax, (title, detections) in zip(comparison_axes, (
        ('Local ridge + neighbor support', detected_peaks_ridge_local),
        ('After long-track validation', detected_peaks_ridge))):
    ax.imshow(visible_data, origin='lower', aspect='auto', cmap='RdBu_r',
              extent=extent, vmin=-limit, vmax=limit, interpolation='none')
    total = 0
    for polarity, (channel_peaks, _) in detections.items():
        times, channels = [], []
        for row, locations in enumerate(channel_peaks):
            times.extend((locations + sample_origin) / sampling_rate)
            channels.extend([PARAMETERS['channel_from'] + row] * len(locations))
        total += len(times)
        ax.scatter(times, channels, s=5, c='#1f77b4' if polarity > 0 else '#2ca02c')
    ax.set(title=f'{title}: {total} original peaks', ylabel='Channel',
           xlim=(WINDOW_START, WINDOW_END))
comparison_axes[-1].set_xlabel('Time from selected file start [s]')
fig.savefig(figure_dir / 'ridge_track_filter_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
STALTA_PARAMETERS = dict(
    sta_seconds=1.0,
    lta_seconds=4.0,
    threshold_on=3.0,
    threshold_off=1.2,
)


def sta_lta_intervals(trace, sampling_rate, parameters):
    """计算经典 STA/LTA，并用高、低阈值确定触发区间 [start, end)。"""
    nsta = round(parameters['sta_seconds'] * sampling_rate)
    nlta = round(parameters['lta_seconds'] * sampling_rate)
    energy_sum = np.r_[0.0, np.cumsum(np.asarray(trace, dtype=float) ** 2)]
    sta, lta = np.zeros(len(trace)), np.zeros(len(trace))
    sta[nsta - 1:] = (energy_sum[nsta:] - energy_sum[:-nsta]) / nsta
    lta[nlta - 1:] = (energy_sum[nlta:] - energy_sum[:-nlta]) / nlta
    ratio = np.zeros(len(trace))
    np.divide(sta, lta, out=ratio, where=lta > 0)
    ratio[:nlta - 1] = 0  # 等长窗积累完整；在带前后补充的数据上计算。

    above = ratio >= parameters['threshold_on']
    below = ratio <= parameters['threshold_off']
    starts = np.flatnonzero(above & np.r_[True, ~above[:-1]])
    ends = np.flatnonzero(below & np.r_[False, ~below[:-1]])
    intervals, previous_end = [], -1
    for start in starts:
        if start < previous_end:  # 尚未关闭的触发只算一个区间。
            continue
        next_end = np.searchsorted(ends, start, side='right')
        end = int(ends[next_end]) if next_end < len(ends) else len(trace)
        intervals.append((int(start), end))
        previous_end = end
    return ratio, np.asarray(intervals, dtype=int).reshape(-1, 2)


def find_channel_peaks_stalta(data, sampling_rate, parameters, stalta_parameters, polarity):
    """用 STA/LTA 触发约束波形峰；同一触发区间可以包含多个车辆峰。"""
    peaks, strengths = [], []
    margin = round(stalta_parameters['sta_seconds'] * sampling_rate)
    for trace in data:
        ratio, intervals = sta_lta_intervals(trace, sampling_rate, stalta_parameters)
        candidates, _ = signal.find_peaks(
            polarity * trace,
            prominence=parameters['peak_prominence'],
            distance=round(parameters['minimum_separation'] * sampling_rate),
            wlen=round(parameters['prominence_window'] * sampling_rate) | 1,
        )
        eligible = np.zeros(len(candidates), dtype=bool)
        for start, end in intervals:
            # 能量触发和波形峰有时间偏移，向两端各扩展一个 STA 窗。
            eligible |= (candidates >= start - margin) & (candidates < end + margin)
        locations = candidates[eligible]
        scores = []
        for location in locations:
            first = max(0, location - margin)
            last = min(len(ratio), location + margin + 1)
            scores.append(np.max(ratio[first:last]))
        peaks.append(locations)
        strengths.append(np.asarray(scores))  # 峰附近的最大 STA/LTA 比值。
    return peaks, strengths


In [ ]:
detected_peaks_stalta = {
    polarity: find_channel_peaks_stalta(
        data, sampling_rate, PARAMETERS, STALTA_PARAMETERS, polarity)
    for polarity in polarities
}
peak_methods = {
    'Prominence': detected_peaks_prominence,
    'Ridge + peaks': detected_peaks_ridge,
    'STA/LTA': detected_peaks_stalta,
}
comparison_first = round(WINDOW_START * sampling_rate) - sample_origin
comparison_last = round(WINDOW_END * sampling_rate) - sample_origin
comparison_data = data[:, comparison_first:comparison_last]
comparison_limit = np.percentile(np.abs(comparison_data), 99)

print('方法          窗口内峰数  有峰通道数  每通道峰数中位数')
fig, axes = plt.subplots(len(peak_methods), 1, figsize=(14, 5 * len(peak_methods)),
                         sharex=True, sharey=True)
for ax, (name, detections) in zip(axes, peak_methods.items()):
    counts = np.zeros(data.shape[0], dtype=int)
    daspy_plot(
        comparison_data, fs=sampling_rate, ax=ax, xmode='channel',
        t0=WINDOW_START, x0=PARAMETERS['channel_from'], transpose=True,
        cmap='RdBu_r', vmin=-comparison_limit, vmax=comparison_limit,
        colorbar_label='Normalized amplitude')
    for polarity, (channel_peaks, _) in detections.items():
        times, channels = [], []
        for row, locations in enumerate(channel_peaks):
            selected = locations[(locations >= comparison_first) & (locations < comparison_last)]
            counts[row] += len(selected)
            times.extend((selected + sample_origin) / sampling_rate)
            channels.extend([PARAMETERS['channel_from'] + row] * len(selected))
        color = 'k' if polarity > 0 else '#2ca02c'
        ax.scatter(times, channels, s=9, c=color, edgecolors='white', linewidths=0.35,
                   zorder=3, label=f'Polarity {polarity:+d}: {len(times)} peaks')
    ax.set(title=f'{name}: {counts.sum()} peaks', xlim=(WINDOW_START, WINDOW_END))
    ax.legend(loc='upper right', fontsize=9, fancybox=False, framealpha=0.95)
    print(f'{name:<12}  {counts.sum():>8}  {np.count_nonzero(counts):>10}'
          f'  {np.median(counts):>16.1f}')
fig.tight_layout()
figure_dir = Path('figures')
figure_dir.mkdir(exist_ok=True)
fig.savefig(figure_dir / 'peak_methods_comparison.png', dpi=250, bbox_inches='tight')
fig.savefig(figure_dir / 'peak_methods_comparison.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# 改这个通道号即可检查其他通道；默认复用预处理图中的通道。
STALTA_PLOT_CHANNEL = PLOT_CHANNEL
stalta_row = STALTA_PLOT_CHANNEL - PARAMETERS['channel_from']
stalta_trace = data[stalta_row]
stalta_ratio, stalta_intervals = sta_lta_intervals(
    stalta_trace, sampling_rate, STALTA_PARAMETERS)
stalta_times = (np.arange(comparison_first, comparison_last) + sample_origin) / sampling_rate

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(stalta_times, stalta_trace[comparison_first:comparison_last],
              color='0.3', linewidth=0.8, label='Waveform')
for (name, detections), color, marker in zip(
        peak_methods.items(), ('#d55e00', '#0072b2'), ('o', '^')):
    for polarity, (channel_peaks, _) in detections.items():
        locations = channel_peaks[stalta_row]
        selected = locations[(locations >= comparison_first) & (locations < comparison_last)]
        axes[0].scatter((selected + sample_origin) / sampling_rate, stalta_trace[selected],
                         s=30, marker=marker, facecolors='none', edgecolors=color,
                         linewidths=1, label=f'{name}, polarity {polarity:+d}', zorder=3)
axes[0].set(title=f'Channel {STALTA_PLOT_CHANNEL}: waveform and picked peaks',
            ylabel='Normalized amplitude')
axes[0].legend(loc='upper right', fontsize=9)
axes[1].plot(stalta_times, stalta_ratio[comparison_first:comparison_last],
              color='k', linewidth=0.8, label='STA/LTA')
axes[1].axhline(STALTA_PARAMETERS['threshold_on'], color='#d55e00',
                linestyle='--', linewidth=1, label='Trigger on')
axes[1].axhline(STALTA_PARAMETERS['threshold_off'], color='#0072b2',
                linestyle='--', linewidth=1, label='Trigger off')
for start, end in stalta_intervals:
    left = max((start + sample_origin) / sampling_rate, WINDOW_START)
    right = min((end + sample_origin) / sampling_rate, WINDOW_END)
    if left < right:
        for ax in axes:
            ax.axvspan(left, right, color='#0072b2', alpha=0.09, linewidth=0)
axes[1].set(title=f"STA {STALTA_PARAMETERS['sta_seconds']:g} s / "
                  f"LTA {STALTA_PARAMETERS['lta_seconds']:g} s; shaded trigger intervals",
            xlabel='Time from file start (s)', ylabel='STA/LTA ratio',
            xlim=(WINDOW_START, WINDOW_END))
axes[1].legend(loc='upper right', fontsize=9)
fig.tight_layout()
fig.savefig(figure_dir / 'stalta_channel_diagnostic.png', dpi=250, bbox_inches='tight')
fig.savefig(figure_dir / 'stalta_channel_diagnostic.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# 复用上一节已计算的正峰；打印实际参数，便于确认当前内核使用的是新版。
print(f"融合找峰：候选突出度={RIDGE_PARAMETERS['candidate_prominence']:g}，"
      f"脊线门槛={RIDGE_PARAMETERS['ridge_threshold']:g}；"
      f"强峰/弱峰邻道支持={RIDGE_PARAMETERS['strong_neighbors']}/"
      f"{RIDGE_PARAMETERS['weak_neighbors']}")
hough_tracks, hough_diagnostics = [], {}
for polarity, (channel_peaks, _) in detected_peaks.items():
    # 复用已完成的全局验证，避免对同一批局部峰重复做霍夫计算。
    picked = ridge_track_candidates[polarity]
    diagnostics = ridge_track_diagnostics[polarity]
    for track in picked:
        track['polarity'] = polarity
    hough_tracks.extend(picked)
    hough_diagnostics[polarity] = diagnostics

hough_tracks.sort(key=lambda track: track['points'][:, 0].min())
print(f'局部霍夫 + 曲线拟合：{len(hough_tracks)} 条候选轨迹')
print('编号  极性  方向        中位速度(km/h)  支持通道数  时间残差RMSE(s)')
for identifier, track in enumerate(hough_tracks, 1):
    direction = '通道递增' if track['direction'] > 0 else '通道递减'
    print(f"{identifier:>3}  {track['polarity']:+d}    {direction}"
          f"  {track['speed'] * 3.6:>10.2f}  {len(track['points']):>10}"
          f"  {track['rmse']:>15.3f}")


In [ ]:
# 左图检查霍夫输入；右图在实际支持范围内绘制曲线，不向两端外推。
hough_first = round(WINDOW_START * sampling_rate) - sample_origin
hough_last = round(WINDOW_END * sampling_rate) - sample_origin
hough_window = data[:, hough_first:hough_last]
hough_extent = [WINDOW_START - 0.5 / sampling_rate, WINDOW_END - 0.5 / sampling_rate,
                PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5]
hough_limit = np.percentile(np.abs(hough_window), 99)
fig, axes = plt.subplots(1, 2, figsize=(22, 8), sharex=True, sharey=True,
                         layout='constrained')
for ax in axes:
    ax.imshow(hough_window, origin='lower', aspect='auto', cmap='RdBu_r',
              interpolation='none', extent=hough_extent,
              vmin=-hough_limit, vmax=hough_limit)
    ax.set(xlabel='Time from selected file start [s]', ylabel='Channel',
           xlim=(WINDOW_START, WINDOW_END))

for polarity, diagnostics in hough_diagnostics.items():
    rows, bins = np.nonzero(diagnostics['peak_image'])
    times = WINDOW_START + (bins + 0.5) * HOUGH_PARAMETERS['time_step']
    axes[0].scatter(times, rows + PARAMETERS['channel_from'], s=5,
                    label=f'Polarity {polarity:+d}: {len(rows)} voting pixels')
axes[0].set_title('Hough input: ridge-validated waveform peaks')
axes[0].legend(loc='upper right')

for identifier, track in enumerate(hough_tracks, 1):
    rows = track['rows']
    curve_rows = np.linspace(rows[0], rows[-1], 300)
    channels = curve_rows + PARAMETERS['channel_from']
    times = track['model'](curve_rows)
    line, = axes[1].plot(times, channels, linewidth=1.6, zorder=4,
                         path_effects=[patheffects.withStroke(linewidth=3.6, foreground='white')],
                         label=f"{identifier}: {track['speed'] * 3.6:.1f} km/h")
    axes[1].scatter(track['points'][:, 0] / sampling_rate, track['points'][:, 1],
                    s=12, color=line.get_color(), edgecolors='white', linewidths=0.45, zorder=5)
axes[1].set_title(f'Local Hough + curve fitting: {len(hough_tracks)} tracks')
if hough_tracks:
    axes[1].legend(loc='upper right', fontsize=9)
plt.show()


In [ ]:
# 两种峰点输入使用相同的局部霍夫与曲线参数；候选轨迹数不等于车辆数。
hough_method_tracks = {}
print('方法          输入峰数  曲线轨迹数  支持通道中位数  覆盖率中位数  RMSE中位数(s)')
fig, axes = plt.subplots(len(peak_methods), 1, figsize=(14, 5 * len(peak_methods)),
                         sharex=True, sharey=True)
for ax, (name, detections) in zip(axes, peak_methods.items()):
    method_tracks = []
    input_peak_count = 0
    for polarity, (channel_peaks, _) in detections.items():
        if name == 'Ridge + peaks':
            picked = ridge_track_candidates[polarity]
            input_peak_count += sum(np.count_nonzero(
                (locations >= first) & (locations < last)) for locations in channel_peaks)
        else:
            picked, diagnostics = pick_hough_tracks(
                channel_peaks, sampling_rate, sample_origin,
                WINDOW_START, WINDOW_END, PARAMETERS, HOUGH_PARAMETERS)
            input_peak_count += np.count_nonzero(diagnostics['peak_image'])
        for track in picked:
            track['polarity'] = polarity
        method_tracks.extend(picked)
    hough_method_tracks[name] = method_tracks
    daspy_plot(
        comparison_data, fs=sampling_rate, ax=ax, xmode='channel',
        t0=WINDOW_START, x0=PARAMETERS['channel_from'], transpose=True,
        cmap='RdBu_r', vmin=-comparison_limit, vmax=comparison_limit,
        colorbar_label='Normalized amplitude')
    for identifier, track in enumerate(method_tracks, 1):
        rows = track['rows']
        curve_rows = np.linspace(rows[0], rows[-1], 300)
        times = track['model'](curve_rows)
        line, = ax.plot(times, curve_rows + PARAMETERS['channel_from'], linewidth=1.6, zorder=4,
                         path_effects=[patheffects.withStroke(linewidth=3.6, foreground='white')],
                         label=f"{identifier}: {track['speed'] * 3.6:.1f} km/h")
        ax.scatter(track['points'][:, 0] / sampling_rate, track['points'][:, 1],
                    s=12, color=line.get_color(), edgecolors='white', linewidths=0.45, zorder=5)
    ax.set(title=f'{name}: {len(method_tracks)} curve tracks',
           xlim=(WINDOW_START, WINDOW_END))
    if method_tracks:
        ax.legend(loc='upper right', fontsize=8, ncol=2, fancybox=False)
        support = np.median([len(track['points']) for track in method_tracks])
        rmse = np.median([track['rmse'] for track in method_tracks])
        coverage = np.median([track['support_fraction'] for track in method_tracks])
        print(f'{name:<12}  {input_peak_count:>8}  {len(method_tracks):>10}'
              f'  {support:>14.1f}  {coverage:>12.1%}  {rmse:>14.3f}')
    else:
        print(f'{name:<12}  {input_peak_count:>8}  {0:>10}')
fig.tight_layout()
fig.savefig(figure_dir / 'hough_peak_methods_comparison.png', dpi=250, bbox_inches='tight')
fig.savefig(figure_dir / 'hough_peak_methods_comparison.pdf', bbox_inches='tight')
plt.show()


## 可选增强与四组对照：f-k 方向分离、Hilbert 包络

原来的 14 个单元格完整保留。下方新增的 4 个代码单元依次完成配置、四组实验、指标导出和绘图，不覆盖原来的 `data`、`detected_peaks` 或 `hough_tracks`。

- 对照组为 Baseline、FK only、Envelope only、FK + envelope，复用相同的脊线、峰间隔、速度与霍夫设置。
- 自动双向模式分别提取通道递增、递减的 f-k 分量，各自检测并按方向跟踪后汇总，不直接删除另一方向的车辆。
- 根据 FFT 关系 `k=-f/v` 确定方向；使用平滑掩膜、加窗和补零，检查共轭对称，不按车辆速度裁剪振动载波。
- Hilbert 沿时间轴提取幅值包络并对称平滑。包络不等同于 STA/LTA，也不保证单峰；包络峰不能解释为原波形极值。
- 新特征按完整上下文的 `1.4826 × MAD` 缩放，复用原突出度门槛。保留全部时间采样点；既有时间峰间隔、邻道支持和轨迹约束用于筛选，每条轨迹每通道最多保留一个观测，不保证每车唯一中心。

**恢复说明：新增单元已补回，当前不标记为重新执行。** 本节末尾直接展示此前实际运行保存的指标与图像；复算时从头执行 Notebook。完整副本与恢复工具在 `review_two_steps/`。


In [ ]:
import json
import csv
import sys
from importlib import reload

_enhancement_dir = next((p for p in (Path.cwd(), Path.cwd() / 'notebooks')
                         if (p / 'vehicle_enhancements.py').is_file()), None)
if _enhancement_dir is None:
    raise FileNotFoundError('请从 notebooks 或项目根目录启动此 Notebook')
if str(_enhancement_dir) not in sys.path:
    sys.path.insert(0, str(_enhancement_dir))
import vehicle_enhancements
reload(vehicle_enhancements)
from vehicle_enhancements import compare_two_steps, comparison_metrics

FK_PARAMETERS = dict(
    padding_seconds=5.0,
    padding_channels=16,
    taper_fraction=0.05,
    transition_bins=2.0,
)
ENVELOPE_PARAMETERS = dict(
    smoothing_seconds=0.25,  # 高斯 sigma，单位 s；加大可能合并相近事件。
    padding_seconds=5.0,
    block_channels=8,
)
comparison_output = _enhancement_dir / 'figures' / 'two_step_comparison'
comparison_output.mkdir(parents=True, exist_ok=True)
legacy_result = dict(
    tracks=hough_tracks,
    detections=detected_peaks_ridge,
    local=detected_peaks_ridge_local,
    raw_candidates=sum(d['raw_candidate_count'] for d in ridge_local_diagnostics.values()),
    local_candidates=sum(d['selected_count'] for d in ridge_local_diagnostics.values()),
    seconds=None,
)


In [ ]:
comparison_results, enhancement_previews = compare_two_steps(
    data, sampling_rate, sample_origin, WINDOW_START, WINDOW_END,
    PARAMETERS, RIDGE_PARAMETERS, HOUGH_PARAMETERS,
    find_channel_peaks_ridge, pick_hough_tracks, filter_ridge_peaks_by_tracks,
    baseline=legacy_result, fk_options=FK_PARAMETERS,
    envelope_options=ENVELOPE_PARAMETERS,
    progress=lambda message: print(message, flush=True),
)
# 单独保留两步组实验输出；原来的主流程变量继续保留。
ENHANCED_METHOD = 'FK + envelope'
enhanced_peaks = comparison_results[ENHANCED_METHOD]['detections']
enhanced_tracks = comparison_results[ENHANCED_METHOD]['tracks']
assert comparison_results['Baseline']['tracks'] is hough_tracks
assert comparison_results['Baseline']['detections'] is detected_peaks_ridge

for method_name, result in comparison_results.items():
    for track in result['tracks']:
        rows = track['rows']
        points = track['points']
        assert np.all(np.diff(rows) > 0)
        assert np.all(np.diff(rows) <= PARAMETERS['maximum_missed_channels'] + 1)
        assert np.all(points[:, 1] == rows + PARAMETERS['channel_from'])
        times = points[:, 0] / sampling_rate
        assert np.all((times >= WINDOW_START) & (times < WINDOW_END))
        assert np.max(np.abs(times - track['model'](rows))) <= HOUGH_PARAMETERS['match_tolerance'] + 1e-9
        assert np.all(track['speed_profile'] <= PARAMETERS['maximum_speed'] + 1e-9)
        assert np.all(track['speed_profile'] >= PARAMETERS['minimum_speed'] - 1e-9)
        assert track['support_fraction'] >= HOUGH_PARAMETERS['minimum_track_density']
        for row, sample in zip(rows, points[:, 0]):
            assert sample - sample_origin in result['detections'][1][0][row]
        feature_direction = track.get('feature_direction', 'auto')
        if feature_direction != 'auto':
            assert track['direction'] == (1 if feature_direction == 'increasing' else -1)
print('检查通过：基线保留；轨迹的观测、缺口、速度、残差和 f-k 分支方向满足约束。')


In [ ]:
comparison_summary, comparison_track_details = comparison_metrics(comparison_results)
print('方法              局部候选  最终峰数  轨迹数  递增/递减  支持通道中位数  覆盖率中位数  RMSE中位数(s)')
for record in comparison_summary:
    support = record['median_support_channels']
    coverage = record['median_coverage']
    rmse = record['median_rmse_seconds']
    support_text = f'{support:.1f}' if support is not None else '--'
    coverage_text = f'{coverage:.1%}' if coverage is not None else '--'
    rmse_text = f'{rmse:.3f}' if rmse is not None else '--'
    print(f"{record['method']:<18} {record['local_candidates']:>7} {record['selected_peaks']:>9}"
          f" {record['tracks']:>6} {record['increasing_tracks']}/{record['decreasing_tracks']}"
          f" {support_text:>17} {coverage_text:>14} {rmse_text:>17}")

comparison_report = dict(
    source_file=str(SOURCE_FILE), sampling_rate=sampling_rate,
    sample_origin=sample_origin, window=[WINDOW_START, WINDOW_END],
    parameters=PARAMETERS, ridge_parameters=RIDGE_PARAMETERS,
    hough_parameters=HOUGH_PARAMETERS, fk_parameters=FK_PARAMETERS,
    envelope_parameters=ENVELOPE_PARAMETERS,
    summary=comparison_summary, track_details=comparison_track_details,
    fk_diagnostics={name: result.get('fk', {}) for name, result in comparison_results.items()},
    feature_scales={name: result.get('scales', {}) for name, result in comparison_results.items()},
    notes=[
        'Original implementation and main tracking outputs retained.',
        'FK directions processed separately and combined after direction-constrained tracking.',
        'New features scaled using 1.4826*MAD over the complete loaded context.',
        'Envelope peaks are feature extrema, not original waveform extrema.',
        'Candidate counts of FK variants sum two direction branches in auto mode.',
        'Runtime excludes legacy baseline and plotting; FK separation shared between FK variants.',
        'Track count, support, coverage and RMSE are consistency metrics, not vehicle accuracy.',
    ],
)
(comparison_output / 'metrics.json').write_text(
    json.dumps(comparison_report, ensure_ascii=False, indent=2), encoding='utf-8')
for filename, records in [('summary.csv', comparison_summary), ('tracks.csv', comparison_track_details)]:
    if records:
        with (comparison_output / filename).open('w', newline='', encoding='utf-8-sig') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(records[0]))
            writer.writeheader()
            writer.writerows(records)

comparison_tracks_export = {}
for name, result in comparison_results.items():
    comparison_tracks_export[name] = [dict(
        points=t['points'].tolist(), rows=t['rows'].tolist(),
        model_coefficients=t['model'].convert().coef.tolist(),
        direction=int(t['direction']), speed_kmh=float(t['speed'] * 3.6),
        rmse_seconds=float(t['rmse']), coverage=float(t['support_fraction']),
    ) for t in result['tracks']]
(comparison_output / 'track_observations.json').write_text(
    json.dumps(comparison_tracks_export, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'指标与逐轨迹观测已保存：{comparison_output}')


In [ ]:
comparison_extent = [WINDOW_START - 0.5 / sampling_rate, WINDOW_END - 0.5 / sampling_rate,
                     PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5]
comparison_background = enhancement_previews['Baseline']['auto']
comparison_vlim = max(float(np.percentile(np.abs(comparison_background), 99)), 1e-8)

# 固定原波形底图、色标、时间和通道范围。
fig, axes = plt.subplots(2, 2, figsize=(22, 13), sharex=True, sharey=True,
                         layout='constrained')
for ax, (name, result), record in zip(axes.ravel(), comparison_results.items(), comparison_summary):
    ax.imshow(comparison_background, origin='lower', aspect='auto', cmap='RdBu_r',
              interpolation='none', extent=comparison_extent,
              vmin=-comparison_vlim, vmax=comparison_vlim, alpha=0.8)
    for peaks, _ in result['local'].values():
        for row, locations in enumerate(peaks):
            times = (locations + sample_origin) / sampling_rate
            inside = (times >= WINDOW_START) & (times < WINDOW_END)
            ax.scatter(times[inside], np.full(np.count_nonzero(inside), row + PARAMETERS['channel_from']),
                       s=3, c='0.25', alpha=0.25, zorder=2)
    for identifier, track in enumerate(result['tracks'], 1):
        curve_rows = np.linspace(track['rows'][0], track['rows'][-1], 300)
        line, = ax.plot(track['model'](curve_rows), curve_rows + PARAMETERS['channel_from'],
                        linewidth=1.5, zorder=4,
                        path_effects=[patheffects.withStroke(linewidth=3, foreground='white')],
                        label=f"{identifier}: {track['speed'] * 3.6:.1f} km/h, n={len(track['rows'])}")
        ax.scatter(track['points'][:, 0] / sampling_rate, track['points'][:, 1],
                   s=9, color=line.get_color(), edgecolors='white', linewidths=0.3, zorder=5)
    ax.set(title=f"{name}: {record['tracks']} tracks, {record['selected_peaks']} selected peaks",
           xlabel='Time from selected file start [s]', ylabel='Channel',
           xlim=(WINDOW_START, WINDOW_END),
           ylim=(PARAMETERS['channel_from'] - 0.5, PARAMETERS['channel_to'] + 0.5))
    if result['tracks']:
        ax.legend(loc='upper right', fontsize=8)
fig.suptitle('Same waveform background and tracking settings; gray = local candidates', fontsize=14)
fig.savefig(comparison_output / 'tracks_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

directions_to_plot = ('increasing', 'decreasing') if PARAMETERS['direction'] == 'auto' else (PARAMETERS['direction'],)
envelope_vlim = max(float(np.percentile(
    enhancement_previews['Envelope only']['auto'], 99)), 1e-8)
fig, axes = plt.subplots(4, len(directions_to_plot), figsize=(22, 19),
                         squeeze=False, sharex=True, sharey=True, layout='constrained')
for row, name in enumerate(comparison_results):
    for col, direction in enumerate(directions_to_plot):
        ax = axes[row, col]
        feature = enhancement_previews[name].get(direction, enhancement_previews[name].get('auto'))
        envelope_plot = name in ('Envelope only', 'FK + envelope')
        artist = ax.imshow(feature, origin='lower', aspect='auto', interpolation='none',
                           extent=comparison_extent, cmap='magma' if envelope_plot else 'RdBu_r',
                           vmin=0 if envelope_plot else -comparison_vlim,
                           vmax=envelope_vlim if envelope_plot else comparison_vlim)
        suffix = direction if name.startswith('FK') else 'shared input'
        ax.set(title=f'{name}: {suffix}', xlabel='Time [s]', ylabel='Channel',
               xlim=(WINDOW_START, WINDOW_END))
        fig.colorbar(artist, ax=ax, fraction=0.025, label='Feature amplitude / robust scale')
fig.savefig(comparison_output / 'features_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('轨迹对照：', comparison_output / 'tracks_comparison.png')
print('特征对照：', comparison_output / 'features_comparison.png')
print('增强实验输出：enhanced_peaks、enhanced_tracks；原输出：detected_peaks、hough_tracks。')


### 已保存的真实数据对照结果（2026-11-02）

来源：`ch1_2026-09-05-13-51-27_2_30720_520_1000.bin`，0–60 秒、291–520 通道。以下为此前完整执行后保存的结果，本次恢复没有重新计算这些结果。

| 方法 | 最终峰数 | 轨迹数（递增/递减） | 支持通道中位数 | 覆盖率中位数 | RMSE 中位数 / s |
|---|---:|---:|---:|---:|---:|
| Baseline | 217 | 3（2/1） | 68 | 51.5% | 0.267 |
| FK only | 104 | 1（0/1） | 104 | 60.8% | 0.244 |
| Envelope only | 82 | 1（0/1） | 82 | 61.2% | 0.256 |
| FK + envelope | 87 | 1（0/1） | 87 | 54.4% | 0.233 |

**此前这组参数下，两步叠加未体现整体改善。** 增强组均未提取出原流程的两条递增候选。共同递减轨迹的支持通道由 95 变为 87，覆盖率由 59.0% 变为 54.4%，最大连续缺失由 10 增至 18 个通道；RMSE 由 0.235 秒略降至 0.233 秒。

原流程继续作为主输出。没有人工真值，不能判定减少的轨迹是真实漏检还是误检被剔除，也不能用较小残差证明定位更准确。

[原执行日志](review_two_steps/execution.log) · [完整指标](figures/two_step_comparison/metrics.json) · [使用说明](review_two_steps/README.md)

四组轨迹对照：

![四组轨迹对照](figures/two_step_comparison/tracks_comparison.png)

两方向的特征对照：

![特征对照](figures/two_step_comparison/features_comparison.png)
